In [ ]:
# Association Rules in Google Colab (Groceries Dataset)
# This notebook:

# Uploads the Excel dataset to Colab
# Converts the data to transactions
# Computes support (frequent itemsets)
# Computes confidence (association rules)
# Computes lift (association rules)
# Parameters: min_support = 0.01, min_confidence = 0.20.

# Install required libraries (run once per session)
!pip -q install mlxtend openpyxl


## Upload the dataset
Upload **Groceries_dataset.xlsx** when prompted.

In [ ]:
# Upload file to Colab
from google.colab import files
uploaded = files.upload()  # choose Groceries_dataset.xlsx
uploaded.keys()


Saving Groceries_dataset.xlsx to Groceries_dataset.xlsx


dict_keys(['Groceries_dataset.xlsx'])

In [ ]:
# Read the Excel file
import pandas as pd

# If you uploaded exactly 'Groceries_dataset.xlsx', this will work as-is.
file_name = 'Groceries_dataset.xlsx'
df = pd.read_excel(file_name)
df.head()


,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,2015-05-01 00:00:00,whole milk
2,2300,19-09-2015,pip fruit
3,1187,2015-12-12 00:00:00,other vegetables
4,3037,2015-01-02 00:00:00,whole milk


In [ ]:
# Quick data check
# We expect at least:

# Member_number
# itemDescription


# Cell 4: Basic cleaning + keep required columns
df = df.copy()
df.columns = [c.strip() for c in df.columns]

required = {'Member_number', 'itemDescription'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {list(df.columns)}")

# Drop empty items
df = df.dropna(subset=['Member_number', 'itemDescription'])
df['itemDescription'] = df['itemDescription'].astype(str).str.strip()
df = df[df['itemDescription'] != '']

df.head(), df.shape


(   Member_number                 Date   itemDescription
 0           1808           21-07-2015    tropical fruit
 1           2552  2015-05-01 00:00:00        whole milk
 2           2300           19-09-2015         pip fruit
 3           1187  2015-12-12 00:00:00  other vegetables
 4           3037  2015-01-02 00:00:00        whole milk,
 (38765, 3))

In [ ]:
# Convert rows into transactions
# A transaction is the set of items purchased by one member. We convert the dataset into a list like:

# Member 1000 → ['whole milk', 'yogurt', ...]
# Then we one-hot encode it for the Apriori algorithm.

# Build transaction list
transactions = df.groupby('Member_number')['itemDescription'].apply(list).tolist()
transactions[:3]


[['soda',
  'canned beer',
  'sausage',
  'sausage',
  'whole milk',
  'whole milk',
  'pickled vegetables',
  'misc. beverages',
  'semi-finished bread',
  'hygiene articles',
  'yogurt',
  'pastry',
  'salty snack'],
 ['frankfurter',
  'frankfurter',
  'beef',
  'sausage',
  'whole milk',
  'soda',
  'curd',
  'white bread',
  'whole milk',
  'soda',
  'whipped/sour cream',
  'rolls/buns'],
 ['tropical fruit',
  'butter milk',
  'butter',
  'frozen vegetables',
  'sugar',
  'specialty chocolate',
  'whole milk',
  'other vegetables']]

In [ ]:
# One-hot encode transactions
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_ary, columns=te.columns_)
basket.head()


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
1,False,False,False,False,False,False,False,False,True,False,...,False,False,False,True,False,True,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [ ]:
# Support
# Support measures how frequently an itemset appears in the dataset.

# We compute frequent itemsets with a minimum support of 0.01 (1%).

# Calculate SUPPORT via frequent itemsets (Apriori)
from mlxtend.frequent_patterns import apriori

min_support = 0.01
itemsets = apriori(basket, min_support=min_support, use_colnames=True)
itemsets = itemsets.sort_values('support', ascending=False)
itemsets.head(10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
113,0.458184,(whole milk)
69,0.376603,(other vegetables)
84,0.349666,(rolls/buns)
94,0.313494,(soda)
114,0.282966,(yogurt)
106,0.233710,(tropical fruit)
85,0.230631,(root vegetables)
7,0.213699,(bottled water)
89,0.206003,(sausage)
1050,0.191380,"(whole milk, other vegetables)"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Confidence
# Confidence measures how often items in the RHS appear in transactions that contain the LHS.

# We generate rules with minimum confidence 0.20 (20%).

# Calculate CONFIDENCE via association rules
from mlxtend.frequent_patterns import association_rules

min_confidence = 0.20
rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)

# Filter for single-item antecedents and consequents
rules_single = rules[
    (rules['antecedents'].apply(lambda x: len(x) == 1)) &
    (rules['consequents'].apply(lambda x: len(x) == 1))
]

# Select and sort
rules_conf = rules_single[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False)


rules_conf.head(10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
2011,(liquor),(whole milk),0.016675,0.631068,1.377325
2885,(mustard),(whole milk),0.014110,0.604396,1.319112
380,(ham),(whole milk),0.036942,0.582996,1.272407
5740,(dog food),(whole milk),0.010005,0.582090,1.270428
2977,(condensed milk),(whole milk),0.013853,0.580645,1.267276
809,(chewing gum),(whole milk),0.025654,0.574713,1.254328
2782,(spread cheese),(whole milk),0.014366,0.571429,1.247160
1488,(dishes),(whole milk),0.019497,0.571429,1.247160
219,(hamburger meat),(whole milk),0.045408,0.565495,1.234211
4346,(canned vegetables),(whole milk),0.011544,0.562500,1.227674


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Lift
**Lift** compares the rule's confidence to what we would expect by chance.

- Lift **> 1**: positive association (items occur together more than expected)
- Lift **≈ 1**: no association
- Lift **< 1**: negative association

In [ ]:
# Calculate LIFT and show top rules by lift
rules_lift = rules_single[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)
rules_lift.head(10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
3773,(specialty bar),(newspapers),0.012314,0.234146,1.674683
3353,(misc. beverages),(frankfurter),0.013084,0.221739,1.612573
1185,(white bread),(whipped/sour cream),0.021550,0.242775,1.569379
4330,(oil),(pork),0.011544,0.207373,1.566552
5367,(red/blush wine),(canned beer),0.010262,0.258065,1.562012
3282,(meat),(domestic eggs),0.013084,0.205645,1.544518
2751,(ice cream),(canned beer),0.014366,0.254545,1.540711
2193,(ham),(canned beer),0.015906,0.251012,1.519325
1315,(sugar),(sausage),0.020523,0.311284,1.511065
3525,(flour),(tropical fruit),0.012827,0.352113,1.506625


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Optional: A clean printable table
Sometimes antecedents/consequents display as `frozenset(...)`. This cell formats them nicely.

In [ ]:
# Cell 10 (Optional): Pretty formatting
def fs_to_str(x):
    return ', '.join(sorted(list(x)))

pretty = rules_lift.copy()
pretty['antecedents'] = pretty['antecedents'].apply(fs_to_str)
pretty['consequents'] = pretty['consequents'].apply(fs_to_str)
pretty.head(15)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
3773,specialty bar,newspapers,0.012314,0.234146,1.674683
3353,misc. beverages,frankfurter,0.013084,0.221739,1.612573
1185,white bread,whipped/sour cream,0.021550,0.242775,1.569379
4330,oil,pork,0.011544,0.207373,1.566552
5367,red/blush wine,canned beer,0.010262,0.258065,1.562012
3282,meat,domestic eggs,0.013084,0.205645,1.544518
2751,ice cream,canned beer,0.014366,0.254545,1.540711
2193,ham,canned beer,0.015906,0.251012,1.519325
1315,sugar,sausage,0.020523,0.311284,1.511065
3525,flour,tropical fruit,0.012827,0.352113,1.506625


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
pretty.to_excel("output_file.xlsx", index=False)

# Trigger a download in Colab
files.download("output_file.xlsx")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag